# Step 0 · SBERT scoring (Colab)

Computes the SBERT similarity between each application and the sector description. It is the part of `1b-DataPreparation` that is too slow on TIP.

- **Input:** `df_sbert_input.parquet`, written by `1b-DataPreparation` (section 2.1).
- **Rule:** the text of an application is its title followed by its abstract; the score is the cosine similarity with `SBERT_DESCRIPTION`. Identical texts are encoded once.
- **Output:** `df_sbert_scored.parquet`, the same applications with one more column, `sbert_similarity`. Upload it into the project folder on TIP and continue `1b-DataPreparation` from section 2.2.

The model and the description are defined here and nowhere else. Use a GPU runtime (Runtime → Change runtime type → GPU).

In [1]:
%pip install -q sentence-transformers pyarrow

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"

# The positive part of the sector definition (first paragraph + "In scope"):
# an embedding model does not understand negation, so the "Out of scope" list
# of DRONE_DESCRIPTION is left out.
SBERT_DESCRIPTION = """
Flying drones: aircraft that fly without a human pilot on board, remotely
piloted or autonomous, of any size, configuration and end use.

In scope:
1. The vehicle: airframe, wings and rotors, propulsion and power supply,
   undercarriage.
2. Flight control, navigation, obstacle avoidance, command and data links,
   ground control stations, swarm coordination.
3. Dedicated infrastructure: launch, landing, recovery, charging or battery
   swap, docking, storage and transport.
4. Airspace and traffic management for unmanned aircraft.
5. Mission equipment and applications, when the invention concerns the
   aircraft itself or the way it is operated in flight (e.g. payload release
   for delivery, spraying controlled through the flight path).
""".strip()

In [2]:
# Upload df_sbert_input.parquet.
# For a large file, copying it from Google Drive is faster than uploading it here.
from google.colab import files

uploaded = files.upload()

Saving df_sbert_input.parquet to df_sbert_input.parquet


In [3]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

df = pd.read_parquet("df_sbert_input.parquet")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Applications:", len(df), "| device:", device)

Applications: 150487 | device: cuda


In [4]:
# Text of each application: title + abstract.
texts = df["appln_title"].fillna("") + ". " + df["appln_abstract"].fillna("")
unique_texts = texts.drop_duplicates().tolist()
print("Distinct texts to encode:", len(unique_texts))

model = SentenceTransformer(SBERT_MODEL, device=device)

embeddings = model.encode(
    unique_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
description_embedding = model.encode(
    SBERT_DESCRIPTION,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Cosine similarity (the vectors are normalised), then back to one row per application.
score_by_text = pd.Series(embeddings @ description_embedding, index=unique_texts)
df["sbert_similarity"] = texts.map(score_by_text).astype("float32").to_numpy()

assert df["sbert_similarity"].notna().all()
df["sbert_similarity"].describe()

Distinct texts to encode: 140666


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.99k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

... (8 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1099 [00:00<?, ?it/s]

,sbert_similarity
count,150487.000000
mean,0.683232
std,0.171956
min,-0.549139
25%,0.608478
50%,0.718148
75%,0.801154
max,0.971339


In [5]:
df.to_parquet("df_sbert_scored.parquet", index=False)
files.download("df_sbert_scored.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>